# Предсказание пола по транзакциям — Sber 2026 DS

**Цель:** ROC AUC > 0.88 и Accuracy > 0.80  
**Решение:** Super-ensemble — все типы фич (685 кандидатов) → top-200 → 5-fold LGB

## 1. Установка зависимостей

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas>=2.3.0", "numpy>=2.2.0", "scikit-learn>=1.7.0",
                "huggingface_hub>=1.9.0", "lightgbm>=4.6.0", "spacy>=3.7.0", "-q"], check=True)
print("Dependencies installed.")

## 2. Загрузка данных

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)

if all(os.path.exists(f"data/{s}.csv") for s in ["train", "val", "test"]):
    print("Data already downloaded.")
else:
    for split in ["train", "val", "test"]:
        cache_path = hf_hub_download(repo_id="mks-logic/gender_prediction",
                                     repo_type="dataset", filename=f"{split}.csv")
        shutil.copy(cache_path, f"data/{split}.csv")
        print(f"  data/{split}.csv downloaded.")

for split in ["train", "val", "test"]:
    size = os.path.getsize(f"data/{split}.csv") // 1024 // 1024
    print(f"  data/{split}.csv  {size} MB")

## 3. Gender-embeddings (spaCy)

Косинусная близость описаний MCC/TR с мужскими/женскими словами.  
Сохраняется в `results/gender_embeddings/`.

In [ ]:
import gc
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

MCC_EMB_CSV = "results/gender_embeddings/mcc_gender.csv"
TR_EMB_CSV  = "results/gender_embeddings/tr_gender.csv"

MALE_WORDS   = ["мужчина","мужской","парень","муж","сын","отец","дедушка",
                "дядя","брат","юноша","мальчик","папа","дед"]
FEMALE_WORDS = ["женщина","женский","девушка","жена","дочь","мать","бабушка",
                "тётя","сестра","девочка","дама","мама","бабуля"]

if os.path.exists(MCC_EMB_CSV) and os.path.exists(TR_EMB_CSV):
    print("Gender embeddings already computed.")
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "ru_core_news_md"], check=True)
    import spacy

    os.makedirs("results/gender_embeddings", exist_ok=True)

    # Читаем уникальные описания (не весь train.csv)
    print("Reading unique code descriptions...", flush=True)
    chunks = pd.read_csv("data/train.csv",
                         usecols=["mcc_code","mcc_code_desc","tr_type","tr_type_desc"],
                         dtype={"mcc_code":"int16","tr_type":"int16"}, chunksize=50_000)
    mcc_seen, tr_seen = {}, {}
    for chunk in chunks:
        for _, row in chunk[["mcc_code","mcc_code_desc"]].drop_duplicates().iterrows():
            if row["mcc_code"] not in mcc_seen:
                mcc_seen[row["mcc_code"]] = row["mcc_code_desc"]
        for _, row in chunk[["tr_type","tr_type_desc"]].drop_duplicates().iterrows():
            if row["tr_type"] not in tr_seen:
                tr_seen[row["tr_type"]] = row["tr_type_desc"]
        if len(mcc_seen) >= 200 and len(tr_seen) >= 80:
            break
    del chunks; gc.collect()

    mcc_map = pd.DataFrame({"mcc_code": list(mcc_seen.keys()), "mcc_code_desc": list(mcc_seen.values())})
    tr_map  = pd.DataFrame({"tr_type":  list(tr_seen.keys()),  "tr_type_desc":  list(tr_seen.values())})
    print(f"  MCC: {len(mcc_map)} codes, TR: {len(tr_map)} types")

    nlp = spacy.load("ru_core_news_md", disable=["parser","ner","tagger","morphologizer","senter"])

    def centroid(words):
        vecs = []
        for w in words:
            tok = nlp(w)
            if tok.has_vector:
                v = tok.vector.copy(); norm = np.linalg.norm(v)
                if norm > 0: vecs.append(v / norm)
        return np.mean(vecs, axis=0) if vecs else np.zeros(nlp.vocab.vectors_length)

    male_vec, female_vec = centroid(MALE_WORDS), centroid(FEMALE_WORDS)

    def embed_text(text):
        if not isinstance(text, str) or not text.strip() or text in ("NaN","н/д"):
            return np.zeros(nlp.vocab.vectors_length)
        doc = nlp(text)
        vecs = [tok.vector / (np.linalg.norm(tok.vector)+1e-9)
                for tok in doc if tok.has_vector and tok.is_alpha]
        if not vecs:
            v = doc.vector; norm = np.linalg.norm(v)
            return v / norm if norm > 0 else np.zeros(nlp.vocab.vectors_length)
        return np.mean(vecs, axis=0)

    def gender_scores(text):
        vec = embed_text(text).reshape(1, -1)
        m = float(cosine_similarity(vec, [male_vec])[0][0])
        f = float(cosine_similarity(vec, [female_vec])[0][0])
        return m, f, m - f

    print("Computing MCC gender scores...", flush=True)
    rows_mcc = []
    for _, r in mcc_map.iterrows():
        m, f, d = gender_scores(r["mcc_code_desc"])
        rows_mcc.append({"mcc_code": r["mcc_code"], "mcc_code_desc": r["mcc_code_desc"],
                         "mcc_male_sim": m, "mcc_female_sim": f, "mcc_gender_diff": d})

    print("Computing TR gender scores...", flush=True)
    rows_tr = []
    for _, r in tr_map.iterrows():
        m, f, d = gender_scores(r["tr_type_desc"])
        rows_tr.append({"tr_type": r["tr_type"], "tr_type_desc": r["tr_type_desc"],
                        "tr_male_sim": m, "tr_female_sim": f, "tr_gender_diff": d})

    mcc_scores = pd.DataFrame(rows_mcc)
    tr_scores  = pd.DataFrame(rows_tr)
    mcc_scores.to_csv(MCC_EMB_CSV, index=False, float_format="%.6f")
    tr_scores.to_csv(TR_EMB_CSV,   index=False, float_format="%.6f")
    print("Saved → results/gender_embeddings/")

mcc_emb_df = pd.read_csv(MCC_EMB_CSV)
tr_emb_df  = pd.read_csv(TR_EMB_CSV)
print(f"MCC embeddings: {len(mcc_emb_df)} codes")
print(f"TR  embeddings: {len(tr_emb_df)} types")
display(mcc_emb_df.head())

## 4. Функции: загрузка данных и генерация признаков

In [ ]:
import time, json
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
import lightgbm as lgb

# ── Константы ────────────────────────────────────────────────────────────────
RESULTS_DIR  = "results/solution_2026_04_15_super_200"
OVERALL_PATH = "results/overall.json"
BRANCH       = "solution_2026_04_15_super_200"
TOP_MCC_N    = 20

HOLIDAYS = {
    "new_year": (1,1), "xmas_orth": (1,7), "feb23": (2,23), "mar8": (3,8),
    "may1": (5,1), "may9": (5,9), "halloween": (10,31),
    "nov7": (11,7), "xmas_west": (12,25), "new_year_e": (12,31),
}

LGB_PARAMS = dict(
    learning_rate=0.02, num_leaves=127, min_child_samples=10,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1,
)

# ── Загрузка ─────────────────────────────────────────────────────────────────
def load_split(split, path="data/"):
    df = pd.read_csv(f"{path}{split}.csv",
                     usecols=["customer_id","tr_datetime","mcc_code",
                               "tr_type","amount","term_id","gender"],
                     dtype={"customer_id":"int32","mcc_code":"int16",
                            "tr_type":"int16","amount":"float32","gender":"int8"})
    df["term_id_missing"] = df["term_id"].isna().astype("int8")
    df.drop(columns=["term_id"], inplace=True)
    df["tr_datetime"] = pd.to_datetime(df["tr_datetime"], errors="coerce")
    df.sort_values(["customer_id","tr_datetime"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# ── Базовые признаки (A) ──────────────────────────────────────────────────────
def generate_features(df, top_mcc=None, top_tr=None):
    g = df.groupby("customer_id", sort=False)
    agg_parts = []
    base = g["amount"].agg(["count","sum","mean","std","min","max"])
    base.columns = ["tx_count","amount_sum","amount_mean","amount_std","amount_min","amount_max"]
    base["amount_q25"]    = g["amount"].quantile(0.25)
    base["amount_q75"]    = g["amount"].quantile(0.75)
    base["amount_median"] = g["amount"].quantile(0.5)
    base["amount_cv"]     = base["amount_std"] / (base["amount_mean"].abs() + 1e-6)
    base["amount_range"]  = base["amount_max"] - base["amount_min"]
    agg_parts.append(base)
    pos = (df[df["amount"]>0].groupby("customer_id",sort=False)["amount"]
           .agg(pos_count="count",pos_sum="sum",pos_mean="mean",pos_std="std"))
    neg = (df[df["amount"]<0].groupby("customer_id",sort=False)["amount"]
           .agg(neg_count="count",neg_sum="sum",neg_mean="mean",neg_std="std"))
    pn = pd.concat([pos,neg],axis=1)
    pn["pos_ratio"]            = pn["pos_count"] / (base["tx_count"]+1e-6)
    pn["pos_neg_amount_ratio"] = pn["pos_sum"] / (pn["neg_sum"].abs()+1e-6)
    agg_parts.append(pn)
    div = g.agg(mcc_nunique=("mcc_code","nunique"), tr_type_nunique=("tr_type","nunique"),
                term_missing_ratio=("term_id_missing","mean"))
    agg_parts.append(div)
    df["hour"]       = df["tr_datetime"].dt.hour.astype("int8")
    df["dow"]        = df["tr_datetime"].dt.dayofweek.astype("int8")
    df["month"]      = df["tr_datetime"].dt.month.astype("int8")
    df["is_weekend"] = (df["dow"]>=5).astype("int8")
    df["is_morning"]   = ((df["hour"]>=6)  & (df["hour"]<12)).astype("int8")
    df["is_afternoon"] = ((df["hour"]>=12) & (df["hour"]<18)).astype("int8")
    df["is_evening"]   = ((df["hour"]>=18) & (df["hour"]<23)).astype("int8")
    df["is_night"]     = ((df["hour"]>=23) | (df["hour"]<6)).astype("int8")
    g2 = df.groupby("customer_id",sort=False)
    temp = g2.agg(hour_mean=("hour","mean"), hour_std=("hour","std"),
                  weekend_ratio=("is_weekend","mean"), morning_ratio=("is_morning","mean"),
                  afternoon_ratio=("is_afternoon","mean"), evening_ratio=("is_evening","mean"),
                  night_ratio=("is_night","mean"), month_nunique=("month","nunique"))
    for d in range(7):
        temp[f"dow{d}_ratio"] = (df["dow"]==d).groupby(df["customer_id"],sort=False).mean()
    agg_parts.append(temp)
    dt_agg = g2["tr_datetime"].agg(["min","max"])
    span = pd.DataFrame(index=dt_agg.index)
    span["active_days"]    = (dt_agg["max"]-dt_agg["min"]).dt.days
    span["tx_per_day"]     = base["tx_count"] / (span["active_days"]+1)
    span["amount_per_day"] = base["amount_sum"] / (span["active_days"]+1)
    df["delta_days"] = g2["tr_datetime"].transform(lambda x: x.diff().dt.total_seconds()/86400)
    span["avg_days_between_tx"] = g2["delta_days"].mean()
    span["std_days_between_tx"] = g2["delta_days"].std()
    df.drop(columns=["delta_days"], inplace=True)
    agg_parts.append(span)
    df["time_frac"] = g2["tr_datetime"].transform(
        lambda x: (x-x.min()).dt.total_seconds() / (max((x.max()-x.min()).total_seconds(),1)))
    df["is_recent"] = (df["time_frac"]>0.75).astype("int8")
    df["is_early"]  = (df["time_frac"]<0.25).astype("int8")
    rec_mean   = df[df["is_recent"]==1].groupby("customer_id",sort=False)["amount"].mean().rename("recent_amount_mean")
    early_mean = df[df["is_early"]==1].groupby("customer_id",sort=False)["amount"].mean().rename("early_amount_mean")
    rec = pd.concat([g2["is_recent"].mean().rename("recent_ratio"),rec_mean,early_mean],axis=1).fillna(0)
    rec["amount_trend"] = rec["recent_amount_mean"] - rec["early_amount_mean"]
    agg_parts.append(rec)
    df.drop(columns=["time_frac","is_recent","is_early"], inplace=True)
    def entropy(series):
        p = series.value_counts(normalize=True)
        return -(p * np.log2(p+1e-9)).sum()
    ent = pd.DataFrame({"mcc_entropy": g["mcc_code"].apply(entropy),
                        "tr_type_entropy": g["tr_type"].apply(entropy)})
    agg_parts.append(ent)
    wknd_mean = df[df["is_weekend"]==1].groupby("customer_id",sort=False)["amount"].mean().rename("weekend_amount_mean")
    wkdy_mean = df[df["is_weekend"]==0].groupby("customer_id",sort=False)["amount"].mean().rename("weekday_amount_mean")
    wknd_df = pd.concat([wknd_mean,wkdy_mean],axis=1).fillna(0)
    wknd_df["wknd_wkdy_diff"] = wknd_df["weekend_amount_mean"] - wknd_df["weekday_amount_mean"]
    agg_parts.append(wknd_df)
    bins = [-np.inf,-10000,-1000,0,1000,10000,np.inf]; labels = ["vn","n","nz","sp","mp","lp"]
    df["amt_bucket"] = pd.cut(df["amount"],bins=bins,labels=labels)
    bkt = df.groupby(["customer_id","amt_bucket"],sort=False,observed=False).size().unstack(fill_value=0)
    bkt = bkt.div(bkt.sum(axis=1),axis=0); bkt.columns = [f"bucket_{c}" for c in bkt.columns]
    agg_parts.append(bkt)
    df.drop(columns=["hour","dow","month","is_weekend","is_morning","is_afternoon","is_evening","is_night","amt_bucket"], inplace=True)
    if top_mcc is None:
        top_mcc = df["mcc_code"].value_counts().nlargest(60).index.tolist()
    mcc_frames = []
    for mcc in top_mcc:
        sub = df[df["mcc_code"]==mcc].groupby("customer_id",sort=False)["amount"]
        mcc_frames.extend([sub.count().rename(f"mcc{mcc}_cnt"), sub.mean().rename(f"mcc{mcc}_mean"), sub.std().rename(f"mcc{mcc}_std")])
    if mcc_frames:
        mcc_df = pd.concat(mcc_frames,axis=1).reindex(base.index).fillna(0)
        for mcc in top_mcc:
            mcc_df[f"mcc{mcc}_share"] = mcc_df[f"mcc{mcc}_cnt"] / (base["tx_count"]+1e-6)
        agg_parts.append(mcc_df)
    if top_tr is None:
        top_tr = df["tr_type"].value_counts().nlargest(25).index.tolist()
    tr_frames = []
    for tr in top_tr:
        sub = df[df["tr_type"]==tr].groupby("customer_id",sort=False)["amount"]
        tr_frames.extend([sub.count().rename(f"tr{tr}_cnt"), sub.mean().rename(f"tr{tr}_mean")])
    if tr_frames:
        tr_df = pd.concat(tr_frames,axis=1).reindex(base.index).fillna(0)
        for tr in top_tr:
            tr_df[f"tr{tr}_share"] = tr_df[f"tr{tr}_cnt"] / (base["tx_count"]+1e-6)
        agg_parts.append(tr_df)
    feats = pd.concat(agg_parts,axis=1).fillna(0)
    feats.reset_index(inplace=True)
    target = df[["customer_id","gender"]].drop_duplicates().set_index("customer_id")
    feats = feats.join(target, on="customer_id")
    X = feats.drop(columns=["customer_id","gender"])
    y = feats["gender"]
    return X, y, top_mcc, top_tr

def compute_category_te(df_train, y_train, col, smoothing=20):
    train_cust_ids = df_train["customer_id"].drop_duplicates().values
    global_m = y_train.mean()
    gender_map = dict(zip(train_cust_ids, y_train.values))
    tx = df_train[["customer_id",col]].copy()
    tx["_g"] = tx["customer_id"].map(gender_map)
    agg = tx.groupby(col)["_g"].agg(["mean","count"])
    agg["te"] = (agg["mean"]*agg["count"]+global_m*smoothing) / (agg["count"]+smoothing)
    return agg["te"], global_m

def apply_te_feature(df, te_map, global_mean, col, feat_name):
    tx = df[["customer_id",col]].copy()
    tx["_te"] = tx[col].map(te_map).fillna(global_mean)
    return tx.groupby("customer_id")["_te"].mean().rename(feat_name)

def add_target_encoding(X_train, y_train, X_val, X_test, df_train, df_val, df_test):
    train_cust_ids = df_train["customer_id"].drop_duplicates().reset_index(drop=True)
    val_cust_ids   = df_val["customer_id"].drop_duplicates().reset_index(drop=True)
    test_cust_ids  = df_test["customer_id"].drop_duplicates().reset_index(drop=True)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for col in ["mcc_code","tr_type"]:
        feat_name = f"{col}_te"
        global_m = y_train.mean()
        oof_te = pd.Series(global_m, index=train_cust_ids)
        for tr_idx, vl_idx in skf.split(train_cust_ids, y_train):
            tr_cust = train_cust_ids.iloc[tr_idx]; vl_cust = train_cust_ids.iloc[vl_idx]
            te_map, gm = compute_category_te(df_train[df_train["customer_id"].isin(tr_cust)], y_train.iloc[tr_idx], col)
            cust_te = apply_te_feature(df_train[df_train["customer_id"].isin(vl_cust)], te_map, gm, col, feat_name)
            oof_te.update(cust_te)
        X_train = X_train.copy(); X_train[feat_name] = oof_te.values
        te_map, gm = compute_category_te(df_train, y_train, col)
        for split_df, split_cids, is_val in [(df_val,val_cust_ids,True),(df_test,test_cust_ids,False)]:
            cust_te = apply_te_feature(split_df, te_map, gm, col, feat_name)
            te_vals = cust_te.reindex(split_cids).fillna(gm).values
            if is_val: X_val = X_val.copy(); X_val[feat_name] = te_vals
            else:      X_test = X_test.copy(); X_test[feat_name] = te_vals
    return X_train, X_val, X_test

# ── Дополнительные признаки ───────────────────────────────────────────────────
def extra_features(df):
    """B: per-hour shares + pos/neg min/max/median"""
    df = df.copy()
    df["tr_datetime"] = pd.to_datetime(df["tr_datetime"], errors="coerce")
    df["hour"] = df["tr_datetime"].dt.hour
    hs = df.groupby(["customer_id","hour"]).size().unstack(fill_value=0).astype("float32")
    hs = hs.div(hs.sum(axis=1), axis=0)
    hs.columns = [f"hour_share_{int(c)}" for c in hs.columns]
    pos = df[df["amount"]>0].groupby("customer_id")["amount"].agg(["min","max","median"])
    pos.columns = ["pos_min","pos_max","pos_median"]
    neg = df[df["amount"]<0].groupby("customer_id")["amount"].agg(["min","max","median"])
    neg.columns = ["neg_min","neg_max","neg_median"]
    return pd.concat([hs,pos,neg], axis=1).fillna(0)

def mcc_interaction_features(df, top_mcc_codes):
    """C+D: MCC × time/percentile + depth"""
    df = df.copy()
    df["tr_datetime"] = pd.to_datetime(df["tr_datetime"], errors="coerce")
    df["hour"] = df["tr_datetime"].dt.hour.astype("int8")
    df["dow"]  = df["tr_datetime"].dt.dayofweek.astype("int8")
    df["is_morning"]   = ((df["hour"]>=6)  & (df["hour"]<12)).astype("int8")
    df["is_afternoon"] = ((df["hour"]>=12) & (df["hour"]<18)).astype("int8")
    df["is_evening"]   = ((df["hour"]>=18) & (df["hour"]<23)).astype("int8")
    df["is_night"]     = ((df["hour"]>=23) | (df["hour"]<6)).astype("int8")
    df["is_weekend"]   = (df["dow"]>=5).astype("int8")
    df["is_neg"]       = (df["amount"]<0).astype("int8")
    total_tx = df.groupby("customer_id").size().rename("_total")
    parts = []
    for mcc in top_mcc_codes:
        sub = df[df["mcc_code"]==mcc]
        if len(sub) == 0: continue
        g = sub.groupby("customer_id")
        mcc_tx = g.size().rename(f"_mcc{mcc}_n")
        parts.append(pd.DataFrame({
            f"mcc{mcc}_morning_ratio":   g["is_morning"].mean().astype("float32"),
            f"mcc{mcc}_afternoon_ratio": g["is_afternoon"].mean().astype("float32"),
            f"mcc{mcc}_evening_ratio":   g["is_evening"].mean().astype("float32"),
            f"mcc{mcc}_night_ratio":     g["is_night"].mean().astype("float32"),
            f"mcc{mcc}_weekend_ratio":   g["is_weekend"].mean().astype("float32"),
            f"mcc{mcc}_q25":    g["amount"].quantile(0.25).astype("float32"),
            f"mcc{mcc}_q75":    g["amount"].quantile(0.75).astype("float32"),
            f"mcc{mcc}_median": g["amount"].median().astype("float32"),
            f"mcc{mcc}_neg_ratio":  g["is_neg"].mean().astype("float32"),
            f"mcc{mcc}_max_amount": g["amount"].max().astype("float32"),
            f"mcc{mcc}_amount_cv":  (g["amount"].std()/(g["amount"].mean().abs()+1e-6)).astype("float32"),
            f"mcc{mcc}_cnt_ratio":  (mcc_tx/total_tx).astype("float32"),
        }))
    return pd.concat(parts, axis=1).fillna(0) if parts else pd.DataFrame()

def _signed_days(dates_ts, month, day):
    year = dates_ts.dt.year.values
    ts   = dates_ts.values.astype("datetime64[D]").astype(np.int64)
    results = np.empty(len(ts), dtype=np.int64)
    for offset in (-1, 0, 1):
        try:
            h = pd.to_datetime({"year": year+offset, "month": month, "day": day}
                               ).values.astype("datetime64[D]").astype(np.int64)
        except Exception:
            continue
        d = ts - h
        if offset == -1: results = d
        else: results = np.where(np.abs(d) < np.abs(results), d, results)
    return results.astype(np.int32)

def holiday_features(df):
    """E: 10 праздников × mean/std/near7"""
    df = df.copy()
    df["tr_datetime"] = pd.to_datetime(df["tr_datetime"], errors="coerce")
    parts = []
    for name, (month, day) in HOLIDAYS.items():
        col = f"_hd_{name}"
        df[col] = _signed_days(df["tr_datetime"], month, day)
        g = df.groupby("customer_id")[col]
        parts.append(pd.DataFrame({
            f"hd_{name}_mean":  g.mean().astype("float32"),
            f"hd_{name}_std":   g.std().fillna(0).astype("float32"),
            f"hd_{name}_near7": g.apply(lambda x: (x.abs()<=7).mean()).astype("float32"),
        }))
    return pd.concat(parts, axis=1)

def gender_emb_features(df, mcc_emb, tr_emb):
    """F: gender embedding cosine similarities"""
    df = df.merge(mcc_emb, on="mcc_code", how="left")
    df = df.merge(tr_emb,  on="tr_type",  how="left")
    for c in ["mcc_male_sim","mcc_female_sim","mcc_gender_diff",
              "tr_male_sim","tr_female_sim","tr_gender_diff"]:
        df[c] = df[c].fillna(0).astype("float32")
    df["abs_amount"] = df["amount"].abs()
    g = df.groupby("customer_id")
    def amt_wt(grp):
        w = grp["abs_amount"].values; d = grp["mcc_gender_diff"].values; s = w.sum()
        return float((w*d).sum()/s) if s>0 else 0.0
    return pd.DataFrame({
        "mcc_gender_diff_mean":   g["mcc_gender_diff"].mean().astype("float32"),
        "mcc_gender_diff_std":    g["mcc_gender_diff"].std().fillna(0).astype("float32"),
        "mcc_male_sim_mean":      g["mcc_male_sim"].mean().astype("float32"),
        "mcc_female_sim_mean":    g["mcc_female_sim"].mean().astype("float32"),
        "mcc_gender_diff_amt_wt": df.groupby("customer_id").apply(amt_wt).astype("float32"),
        "tr_gender_diff_mean":    g["tr_gender_diff"].mean().astype("float32"),
        "tr_gender_diff_std":     g["tr_gender_diff"].std().fillna(0).astype("float32"),
        "tr_male_sim_mean":       g["tr_male_sim"].mean().astype("float32"),
        "tr_female_sim_mean":     g["tr_female_sim"].mean().astype("float32"),
    })

def attach(X, feat_df, cust_order):
    if feat_df is None or feat_df.empty: return X
    aligned = feat_df.reindex(cust_order).fillna(0).reset_index(drop=True)
    new_cols = [c for c in aligned.columns if c not in X.columns]
    return pd.concat([X.reset_index(drop=True), aligned[new_cols]], axis=1)

print("Functions defined.")

## 5. Генерация признаков

In [ ]:
mcc_emb = pd.read_csv(MCC_EMB_CSV)[["mcc_code","mcc_male_sim","mcc_female_sim","mcc_gender_diff"]]
tr_emb  = pd.read_csv(TR_EMB_CSV)[["tr_type","tr_male_sim","tr_female_sim","tr_gender_diff"]]
mcc_emb["mcc_code"] = mcc_emb["mcc_code"].astype("int16")
tr_emb["tr_type"]   = tr_emb["tr_type"].astype("int16")

print("Loading splits...")
train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")

top_mcc_codes = train_df["mcc_code"].value_counts().nlargest(TOP_MCC_N).index.tolist()

print("Generating base features (A)...")
X_train, y_train, top_mcc, top_tr = generate_features(train_df)
X_val,   y_val,   _,       _      = generate_features(val_df,  top_mcc=top_mcc, top_tr=top_tr)
X_test,  y_test,  _,       _      = generate_features(test_df, top_mcc=top_mcc, top_tr=top_tr)

print("Adding target encoding...")
X_train, X_val, X_test = add_target_encoding(X_train, y_train, X_val, X_test, train_df, val_df, test_df)

print("Computing extra features (B)...")
b_tr, b_vl, b_te = extra_features(train_df), extra_features(val_df), extra_features(test_df)

print(f"Computing MCC interactions + depth (C+D, top-{TOP_MCC_N})...")
cd_tr = mcc_interaction_features(train_df, top_mcc_codes)
cd_vl = mcc_interaction_features(val_df,   top_mcc_codes)
cd_te = mcc_interaction_features(test_df,  top_mcc_codes)

print("Computing holiday features (E)...")
e_tr, e_vl, e_te = holiday_features(train_df), holiday_features(val_df), holiday_features(test_df)

print("Computing gender embedding features (F)...")
f_tr = gender_emb_features(train_df, mcc_emb, tr_emb)
f_vl = gender_emb_features(val_df,   mcc_emb, tr_emb)
f_te = gender_emb_features(test_df,  mcc_emb, tr_emb)

cust_train = train_df["customer_id"].drop_duplicates().values
cust_val   = val_df["customer_id"].drop_duplicates().values
cust_test  = test_df["customer_id"].drop_duplicates().values
del train_df, val_df, test_df; gc.collect()

for extra_f, vl, te, co_tr, co_vl, co_te in [
    (b_tr, b_vl, b_te, cust_train, cust_val, cust_test),
    (cd_tr,cd_vl,cd_te,cust_train, cust_val, cust_test),
    (e_tr, e_vl, e_te, cust_train, cust_val, cust_test),
    (f_tr, f_vl, f_te, cust_train, cust_val, cust_test),
]:
    X_train = attach(X_train, extra_f, co_tr)
    X_val   = attach(X_val,   vl,      co_vl)
    X_test  = attach(X_test,  te,      co_te)

del b_tr,b_vl,b_te, cd_tr,cd_vl,cd_te, e_tr,e_vl,e_te, f_tr,f_vl,f_te; gc.collect()

candidates = list(X_train.columns)
print(f"Total candidate features: {len(candidates)}")

## 6. Отбор признаков (probe) и обучение ансамбля

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Step 1: отбор top-200
print("Step 1: feature selection...")
probe = lgb.LGBMClassifier(n_estimators=3000, random_state=0, **LGB_PARAMS)
probe.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(period=200)])
best_iter = int(probe.best_iteration_ * 1.1)

fi = pd.Series(probe.feature_importances_, index=candidates).sort_values(ascending=False)
top200 = fi.head(200).index.tolist()

extra_names = set([f"hour_share_{h}" for h in range(24)] +
                  ["pos_min","pos_max","pos_median","neg_min","neg_max","neg_median"])
int_names   = set(f"mcc{m}_{s}" for m in top_mcc_codes
                  for s in ("morning_ratio","afternoon_ratio","evening_ratio","night_ratio","weekend_ratio","q25","q75","median"))
depth_names = set(f"mcc{m}_{s}" for m in top_mcc_codes
                  for s in ("neg_ratio","max_amount","amount_cv","cnt_ratio"))
hol_names   = set(f"hd_{n}_{s}" for n in HOLIDAYS for s in ("mean","std","near7"))
gemb_names  = {"mcc_gender_diff_mean","mcc_gender_diff_std","mcc_male_sim_mean",
               "mcc_female_sim_mean","mcc_gender_diff_amt_wt",
               "tr_gender_diff_mean","tr_gender_diff_std","tr_male_sim_mean","tr_female_sim_mean"}
known = extra_names | int_names | depth_names | hol_names | gemb_names

groups = {"A_base":      [f for f in top200 if f not in known],
          "B_extra":     [f for f in top200 if f in extra_names],
          "C_mcc_time":  [f for f in top200 if f in int_names],
          "D_mcc_depth": [f for f in top200 if f in depth_names],
          "E_holiday":   [f for f in top200 if f in hol_names],
          "F_gender_emb":[f for f in top200 if f in gemb_names]}

print(f"  best_iter (×1.1): {best_iter}")
print("  Feature groups in top-200:")
for grp, feats in groups.items():
    print(f"    {grp}: {len(feats)}")

fi_df = pd.DataFrame({"feature": fi.index, "importance": fi.values})
fi_df.to_csv(f"{RESULTS_DIR}/feature_importance_all.csv", index=False)

X_train_s = X_train[top200]; X_val_s = X_val[top200]; X_test_s = X_test[top200]
del X_train, X_val, X_test; gc.collect()

# Step 2: 5-fold ensemble
print("\nStep 2: 5-fold ensemble on top-200...")
X_full = pd.concat([X_train_s, X_val_s]).reset_index(drop=True)
y_full = pd.concat([y_train, y_val]).reset_index(drop=True)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_probas, models = [], []

t0 = time.perf_counter()
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_full, y_full)):
    m = lgb.LGBMClassifier(n_estimators=best_iter, random_state=fold, **LGB_PARAMS)
    m.fit(X_full.iloc[tr_idx], y_full.iloc[tr_idx], callbacks=[lgb.log_evaluation(period=-1)])
    oof_auc = roc_auc_score(y_full.iloc[vl_idx], m.predict_proba(X_full.iloc[vl_idx])[:, 1])
    test_probas.append(m.predict_proba(X_test_s)[:, 1])
    models.append(m)
    print(f"  fold {fold+1}/5  OOF AUC: {oof_auc:.4f}")
train_time = round(time.perf_counter() - t0, 1)

y_pred_proba = np.mean(test_probas, axis=0)
y_pred = (y_pred_proba >= 0.5).astype(int)
print(f"  train_time: {train_time}s")

## 7. Итоговые метрики

In [ ]:
auc  = round(roc_auc_score(y_test, y_pred_proba), 4)
acc  = round(accuracy_score(y_test, y_pred), 4)
prec = round(precision_score(y_test, y_pred), 4)
rec  = round(recall_score(y_test, y_pred), 4)

print("=" * 40)
print(f"  Model:     super_200  (200 features)")
print(f"  ROC AUC:   {auc}  {'✓' if auc > 0.88 else '✗'} (target > 0.88)")
print(f"  Accuracy:  {acc}  {'✓' if acc > 0.80 else '✗'} (target > 0.80)")
print(f"  Precision: {prec}")
print(f"  Recall:    {rec}")
print(f"  Train:     {train_time}s")
print("=" * 40)

## 7.1. Предсказания на тестовой выборке

In [ ]:
import joblib

# Сохранить модели
model_path = f"{RESULTS_DIR}/models.pkl"
joblib.dump({"models": models, "feature_names": top200, "best_iter": best_iter}, model_path)
print(f"Saved models → {model_path}")

# Сохранить предсказания
test_raw = pd.read_csv("data/test.csv")
customer_pred = pd.DataFrame({
    "customer_id":       cust_test,
    "prediction":        y_pred,
    "prediction_proba":  y_pred_proba.round(6),
})
test_out = test_raw.merge(customer_pred, on="customer_id", how="left")
pred_path = f"{RESULTS_DIR}/test_predictions.csv"
test_out.to_csv(pred_path, index=False)
print(f"Saved predictions → {pred_path}")

display(test_out[["customer_id","gender","prediction","prediction_proba"]].head(10))

## 8. Сохранение результатов

In [ ]:
os.makedirs(os.path.dirname(OVERALL_PATH), exist_ok=True)
overall = json.load(open(OVERALL_PATH)) if os.path.exists(OVERALL_PATH) else []
overall = [e for e in overall if e["branch"] != BRANCH]
overall.append({
    "branch": BRANCH, "date": "2026-04-25",
    "metrics": {"auc_score": auc, "accuracy_score": acc, "precision_score": prec, "recall_score": rec},
    "timing": {"train_time_s": train_time},
    "n_features": 200,
    "feature_groups_in_top200": {k: len(v) for k, v in groups.items()},
    "total_candidates": len(candidates),
})
with open(OVERALL_PATH, "w", encoding="utf-8") as f:
    json.dump(overall, f, ensure_ascii=False, indent=2)
print(f"Updated: {OVERALL_PATH}")

## 9. Сравнение всех экспериментов

In [ ]:
with open(OVERALL_PATH) as f:
    overall = json.load(f)

rows = []
for e in overall:
    m = e["metrics"]
    rows.append({
        "branch":     e["branch"].replace("solution_2026_04_15_", ""),
        "AUC":        m["auc_score"],
        "Accuracy":   m["accuracy_score"],
        "n_features": e.get("n_features", ""),
        "train_s":    e.get("timing", {}).get("train_time_s", ""),
    })

df_res = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
df_res["AUC ✓"] = df_res["AUC"].apply(lambda x: "✓" if x > 0.88 else "")
df_res["Acc ✓"] = df_res["Accuracy"].apply(lambda x: "✓" if x > 0.80 else "")
display(df_res)